# U5 — Calculus for ML: Lab

**How models learn — by following the gradient downhill** — derivatives · partial derivatives · the gradient · chain rule · backpropagation · Hessian

_Day 2 · Phase B — Mathematical Foundations · Bridge unit (Module 1.3.8)_

#objectives

By the end of this lab you will be able to:

Compute derivatives numerically (finite difference) and symbolically (SymPy)

Take partial derivatives and assemble the gradient of a multivariable function

Apply the chain rule by hand and verify it with SymPy

Run a forward and backward pass through a 2-layer network and derive its gradients

Compute a Hessian, and see how a gradient-descent step uses the gradient

#how to use this lab

Each section has two kinds of cells:

Worked demo cells — run them top to bottom and read the comments to learn the pattern.

LAB EXERCISE cells (marked 🧪) — your turn. Replace each `# YOUR CODE HERE` with working code.

Run cells with **Shift + Enter**. Run the demos before attempting the exercises.

In [ ]:
# Core imports for the whole lab
import numpy as np
import sympy as sp

x, y = sp.symbols('x y')      # symbolic variables we'll reuse
sp.init_printing()            # pretty-print symbolic math
np.random.seed(42)
print('Setup complete. SymPy', sp.__version__, '| NumPy', np.__version__)

Setup complete. SymPy 1.14.0 | NumPy 2.0.2


#1. Derivatives — the slope of a function

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. NUMERICAL DERIVATIVE (finite difference)
# -----------------------------------------------------------

# The derivative is the slope: how much f changes for a tiny step h
def f(x):
    return x**2

h = 1e-6
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 6)')

Numerical f'(3): 6.0  (exact = 6)


In [ ]:
# -----------------------------------------------------------
# 🔹 1B. SYMBOLIC DERIVATIVE (SymPy)
# -----------------------------------------------------------

expr = x**2
deriv = sp.diff(expr, x)          # differentiate w.r.t. x
print('d/dx (x**2) =', deriv)     # 2*x

# Evaluate the symbolic derivative at x = 3
print('Symbolic f\'(3) =', deriv.subs(x, 3))

d/dx (x**2) = 2*x
Symbolic f'(3) = 6


#### 🧪 LAB EXERCISE 1 — Numerical vs symbolic derivative

For the function `g(x) = x**3 + 2*x`:
1. Compute the **numerical** derivative at `x = 2` using a finite difference.
2. Compute the **symbolic** derivative with `sp.diff` and print it.
3. Evaluate the symbolic derivative at `x = 2` and confirm it matches step 1.

In [ ]:
def g(x):
    return x**3 + 2*x

# 1. Numerical derivative at x = 2 (use h = 1e-6)
# YOUR CODE HERE
h = 1e-6
slope_at_2 = (g(2 + h) - g(2)) / h
print( round(slope_at_2, 4))

# 2. Symbolic derivative of x**3 + 2*x
# YOUR CODE HERE
expr = x**3 + 2*x
deriv = sp.diff(expr, x)
print(deriv)

# 3. Evaluate the symbolic derivative at x = 2 and compare
# YOUR CODE HERE
print(deriv.subs(x , 2))

14.0
3*x**2 + 2
14


#2. Partial derivatives & the gradient

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. PARTIAL DERIVATIVES
# -----------------------------------------------------------

f2 = x**2 + 3*x*y + y**2

# A partial derivative differentiates ONE variable, holding others fixed
print('df/dx =', sp.diff(f2, x))     # 2*x + 3*y
print('df/dy =', sp.diff(f2, y))     # 3*x + 2*y

df/dx = 2*x + 3*y
df/dy = 3*x + 2*y


In [ ]:
# -----------------------------------------------------------
# 🔹 2B. THE GRADIENT (vector of partials)
# -----------------------------------------------------------

# The gradient stacks every partial derivative into one vector
grad = [sp.diff(f2, v) for v in (x, y)]
print('grad f =', grad)

# Evaluate the gradient at the point (x=1, y=2)
grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad f at (1, 2) =', grad_at)   # points in the steepest-ascent direction

grad f = [2*x + 3*y, 3*x + 2*y]
grad f at (1, 2) = [8, 7]


#### 🧪 LAB EXERCISE 2 — Compute a gradient

For `h2 = x**2 * y + sp.sin(y)`:
1. Compute `dh/dx` and `dh/dy` with `sp.diff`.
2. Assemble the gradient as a list `[dh/dx, dh/dy]`.
3. Evaluate the gradient at the point `(x=2, y=0)`.

In [ ]:
h2 = x**2 * y + sp.sin(y)

# 1. dh/dx and dh/dy
# YOUR CODE HERE
print(sp.diff(h2 , x))
print(sp.diff(h2 , y))

# 2. Assemble the gradient list
# YOUR CODE HERE
grad = [sp.diff(h2 , x) , sp.diff(h2 , y)]
print(grad)

# 3. Evaluate at (x=2, y=0)  -> hint: .subs({x: 2, y: 0})
# YOUR CODE HERE
grad_at = [g.subs({x: 2, y: 0}) for g in grad]
print(grad_at)

2*x*y
x**2 + cos(y)
[2*x*y, x**2 + cos(y)]
[0, 5]


#3. The chain rule

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. CHAIN RULE BY HAND vs SymPy
# -----------------------------------------------------------

# y = sin(x**2) is a composition: outer = sin(u), inner = u = x**2
# Chain rule:  dy/dx = cos(u) * du/dx = cos(x**2) * 2x
by_hand = sp.cos(x**2) * 2*x
by_sympy = sp.diff(sp.sin(x**2), x)

print('By hand :', by_hand)
print('By SymPy:', by_sympy)
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)

By hand : 2*x*cos(x**2)
By SymPy: 2*x*cos(x**2)
Match?   True


In [ ]:
# -----------------------------------------------------------
# 🔹 3B. CHAINING THREE FUNCTIONS
# -----------------------------------------------------------

# y = (3x + 1)**4  -> outer^4, inner (3x+1)
expr3 = (3*x + 1)**4
print('d/dx (3x+1)^4 =', sp.diff(expr3, x))   # 12*(3x+1)^3

d/dx (3x+1)^4 = 12*(3*x + 1)**3


#### 🧪 LAB EXERCISE 3 — Chain rule

For `y = sp.exp(x**2 + 1)`:
1. Write the chain-rule result **by hand** as a SymPy expression (outer derivative × inner derivative).
2. Differentiate the same expression with `sp.diff`.
3. Confirm the two match using `sp.simplify(by_hand - by_sympy) == 0`.

In [ ]:
# y = exp(x**2 + 1);  inner = x**2 + 1, inner' = 2x, outer' = exp(inner)

# 1. By hand (as a SymPy expression)
# YOUR CODE HERE
by_hand = sp.exp(x**2 + 1) * (2*x)
print(by_hand)

# 2. With sp.diff
# YOUR CODE HERE
by_sympy = sp.diff(sp.exp(x**2 + 1), x)
print(by_sympy)

# 3. Confirm they match
# YOUR CODE HERE
print(sp.simplify(by_hand - by_sympy) == 0)

2*x*exp(x**2 + 1)
2*x*exp(x**2 + 1)
True


#4. Backpropagation — the chain rule in a network

A 2-layer network is just a composition:  **x → (W1) → ReLU → (W2) → y_hat**. Backprop applies the chain rule from the loss backwards to every weight.

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. FORWARD PASS
# -----------------------------------------------------------

# Tiny toy problem: 4 samples, 3 input features, 5 hidden units, 1 output
X = np.random.randn(4, 3)
Y = np.random.randn(4, 1)
W1 = np.random.randn(3, 5) * 0.1
W2 = np.random.randn(5, 1) * 0.1

z1 = X @ W1              # linear layer 1
h  = np.maximum(0, z1)      # ReLU activation
y_hat = h @ W2              # linear layer 2 (prediction)
loss = ((y_hat - Y) ** 2).mean()
print('Initial loss:', round(loss, 4))

[[ 0.19339886  0.10673748 -0.27755938 -0.06678051  0.02841912]
 [-0.05511934  0.01600505  0.10626125  0.00618953 -0.03146526]
 [ 0.18071304  0.15981338 -0.16143109 -0.19686505  0.02594252]
 [-0.05398177 -0.1857441  -0.03325938  0.09840611  0.06135041]]
Initial loss: 0.8583


In [ ]:
# -----------------------------------------------------------
# 🔹 4A. FORWARD PASS
# -----------------------------------------------------------

# Tiny toy problem: 4 samples, 3 input features, 5 hidden units, 1 output
X = np.random.randn(4, 3)

Y = np.random.randn(4, 1)

W1 = np.random.randn(3, 5) * 0.1

W2 = np.random.randn(5, 1) * 0.1

#here @ is matrix multiplication
z1 = X @ W1              # linear layer 1
print("z1:",z1)
h  = np.maximum(0, z1)      # ReLU activation
print("h:",h)
y_hat = h @ W2              # linear layer 2 (prediction)
print("y_hat:",y_hat)
loss = ((y_hat - Y) ** 2).mean()

print('Initial loss:', round(loss, 4))

X: [[-0.77282521 -0.23681861 -0.48536355]
 [ 0.08187414  2.31465857 -1.86726519]
 [ 0.68626019 -1.61271587 -0.47193187]
 [ 1.0889506   0.06428002 -1.07774478]]
Y : [[-0.71530371]
 [ 0.67959775]
 [-0.73036663]
 [ 0.21645859]]
W1: [[ 0.00455718 -0.06516003  0.21439441  0.0633919  -0.20251426]
 [ 0.01864543 -0.06617865  0.08524333 -0.07925207 -0.01147364]
 [ 0.05049873  0.08657552 -0.12002964 -0.03345012 -0.04749453]]
W2: [[-0.06533292]
 [ 0.17654542]
 [ 0.04049817]
 [-0.1260884 ]
 [ 0.09178619]]
z1: [[-0.03244773  0.02400905 -0.1276186  -0.01398702  0.18227741]
 [-0.05076359 -0.32017535  0.43898974 -0.11579108  0.04554664]
 [-0.05077433  0.02115287  0.06630288  0.1871006  -0.09805956]
 [-0.04826366 -0.16851634  0.36830568  0.09998702 -0.17007857]]
h: [[0.         0.02400905 0.         0.         0.18227741]
 [0.         0.         0.43898974 0.         0.04554664]
 [0.         0.02115287 0.06630288 0.1871006  0.        ]
 [0.         0.         0.36830568 0.09998702 0.        ]]
y_hat: [

In [ ]:
# -----------------------------------------------------------
# 🔹 4B. BACKWARD PASS (gradients via the chain rule)
# -----------------------------------------------------------

# Work backwards from the loss, one link at a time
dy   = 2 * (y_hat - Y) / Y.size      # d loss / d y_hat
print(dy)
dW2  = h.T @ dy                      # d loss / d W2
print(dW2)
dh   = dy @ W2.T                     # d loss / d h
dz1  = dh * (z1 > 0)                 # ReLU gradient (1 where z1>0 else 0)
dW1  = X.T @ dz1                     # d loss / d W1

print('dW1 shape:', dW1.shape, '(matches W1)')
print('dW2 shape:', dW2.shape, '(matches W2)')

[[ 0.36813647]
 [-0.32881946]
 [ 0.3565975 ]
 [-0.10707504]]
[[0.         0.         0.         0.        ]
 [0.02400905 0.         0.02115287 0.        ]
 [0.         0.43898974 0.06630288 0.36830568]
 [0.         0.         0.1871006  0.09998702]
 [0.18227741 0.04554664 0.         0.        ]]
[[ 0.        ]
 [ 0.01638167]
 [-0.16014127]
 [ 0.05601349]
 [ 0.05212634]]
dW1 shape: (3, 5) (matches W1)
dW2 shape: (5, 1) (matches W2)


In [ ]:
# -----------------------------------------------------------
# 🔹 4C. ONE GRADIENT-DESCENT STEP SHOULD LOWER THE LOSS
# -----------------------------------------------------------

lr = 0.1
W1 -= lr * dW1               # step downhill
W2 -= lr * dW2

# Recompute the loss after the update
h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')

#### 🧪 LAB EXERCISE 4 — Derive the gradients for a 2-layer network

Fresh weights are set up below. Using the chain rule (reuse the pattern from 4A/4B):
1. Run the **forward pass**: compute `z1`, `h` (ReLU), `y_hat`, and `loss`.
2. Run the **backward pass**: compute `dy`, `dW2`, `dh`, `dz1`, `dW1`.
3. Take one gradient-descent step (`lr = 0.05`) and print the loss before and after.

In [ ]:
Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1     # input -> hidden
Wb = np.random.randn(8, 1) * 0.1     # hidden -> output

# 1. Forward pass: z1, h = ReLU(z1), y_hat, loss
# YOUR CODE HERE

# 2. Backward pass: dy, dWb, dh, dz1, dWa
# YOUR CODE HERE

# 3. One step with lr = 0.05; print loss before and after
# YOUR CODE HERE

#5. Hessian & gradient descent

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. THE HESSIAN (matrix of second derivatives = curvature)
# -----------------------------------------------------------

f5 = x**2 + 3*x*y + y**2
H = sp.hessian(f5, (x, y))
print('Hessian of f:')
sp.pprint(H)        # [[2, 3], [3, 2]]

In [ ]:
# -----------------------------------------------------------
# 🔹 5B. GRADIENT DESCENT ON A SIMPLE FUNCTION
# -----------------------------------------------------------

# Minimise f(x) = (x - 4)**2, whose minimum is at x = 4.
# Update rule:  x <- x - lr * f'(x),  with f'(x) = 2*(x - 4)
xv = 0.0          # starting point
lr = 0.2
for step in range(15):
    grad = 2 * (xv - 4)        # the derivative
    xv = xv - lr * grad        # step against the gradient
print('Converged x:', round(xv, 3), ' (true minimum = 4)')

#### 🧪 LAB EXERCISE 5 — Hessian + gradient descent

1. Compute the **Hessian** of `f = x**4 + y**2` with `sp.hessian`.
2. Run **gradient descent** to minimise `f(x) = (x - 7)**2`:
   start at `x = 0`, `lr = 0.1`, 20 steps, using `f'(x) = 2*(x - 7)`. Print the final `x` (should approach 7).

In [ ]:
# 1. Hessian of x**4 + y**2
# YOUR CODE HERE

# 2. Gradient descent to minimise (x - 7)**2
# xv = 0.0; lr = 0.1
# for step in range(20):
#     grad = ...
#     xv = ...
# print('Final x:', xv)
# YOUR CODE HERE

#📘 Summary — Calculus for ML toolkit

| Concept | What it means | Key calls |
| ------- | ------------- | --------- |
| **Derivative** | slope / rate of change | finite difference · `sp.diff` |
| **Partial derivative** | slope along one variable | `sp.diff(f, x)` |
| **Gradient** | vector of all partials → steepest ascent | `[sp.diff(f, v) for v in vars]` |
| **Chain rule** | differentiate compositions | outer' × inner' (SymPy automates) |
| **Backpropagation** | chain rule through a network | forward pass + backward pass |
| **Hessian** | second derivatives → curvature | `sp.hessian(f, vars)` |
| **Gradient descent** | step downhill | `x -= lr * grad` |

**Homework (self-paced):** numerical vs symbolic derivative · compute ∂f/∂x, ∂f/∂y and assemble ∇f · chain-rule derivation + SymPy check · derive gradients for a 2-layer network · compute a Hessian.

**Next:** U6 Probability & Statistics (Part 1). The full optimization machinery — learning rates, SGD vs Adam, convergence — arrives in U13.